#Build silver_grid_events

We transform `bronze_grid_events` into a clean Silver dataset.

## Focus:
- Parse timestamps correctly
- Create event_day field
- Normalize severity and event types
- Clean duration values
- Apply a severity band UDF
- Use df.transform() for modular design

In [0]:
import yaml
from pyspark.sql import functions as F
from pyspark.sql.types import StringType
from pyspark.sql.functions import udf

In [0]:
config_path = "/Workspace/Repos/Mini Projects/Vattenfall-Engineering-Capstone-Project/config/project_config.yml"

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["catalog"]
raw_schema = config["schemas"]["raw"]
refined_schema = config["schemas"]["refined"]

In [0]:
df_events = spark.table(f"{catalog}.{raw_schema}.bronze_grid_events")

display(df_events)

In [0]:
def clean_events(df):
    return (
        df
        .withColumn("region", F.upper("region"))
        .withColumn("event_type", F.upper("event_type"))
        .withColumn("severity", F.upper("severity"))
        .withColumn("event_ts", F.to_timestamp("event_ts"))
        .withColumn("last_updated_ts", F.to_timestamp("last_updated_ts"))
        .withColumn("duration_minutes", F.col("duration_minutes").cast("int"))
        .withColumn("event_day", F.to_date("event_ts"))
    )

df_events = df_events.transform(clean_events)

We classify severity into business-friendly categories.

In [0]:
def severity_band(sev):
    if sev == "LOW":
        return "MINOR"
    elif sev == "MEDIUM":
        return "MODERATE"
    else:
        return "SEVERE"

severity_band_udf = udf(severity_band, StringType())

df_events = df_events.withColumn(
    "severity_band",
    severity_band_udf(F.col("severity"))
)

In [0]:
df_events = df_events.filter(
    F.col("event_ts").isNotNull() &
    F.col("severity").isNotNull() &
    (F.col("duration_minutes") > 0)
)

In [0]:
df_events = df_events.drop(
    *[c for c in df_events.columns if "rescued" in c.lower()]
)

In [0]:
display(df_events)
df_events.printSchema()

print("Row count:", df_events.count())

In [0]:
df_events.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable(f"{catalog}.{refined_schema}.silver_grid_events")

In [0]:
display(
    spark.table(f"{catalog}.{refined_schema}.silver_grid_events")
)